# Tutorial: Power BI Measure Validation Showcase

This notebook is a guided demo for discovering a Power BI dataset, loading its measure validation template, executing selected validation cases, reviewing report context, and saving organized outputs under `test-results/`.


## Demo Flow

1. Authenticate with the repo's existing Power BI auth pattern.
2. List workspaces and choose one for the demo.
3. List datasets and choose the target semantic model.
4. Load the dataset's measure validation template or generated candidate file.
5. Filter and select validation cases.
6. Execute visible DAX queries and preview returned data.
7. Review related report metadata.
8. Save a clean result package for the run.


In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Markdown, display

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'src').exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from scripts.helpers.measure_validation_demo import (
    build_execution_summary_frame,
    build_report_context,
    build_report_summary_markdown,
    execute_validation_case,
    filter_validation_cases,
    get_datasets_frame,
    get_reports_frame,
    get_workspaces_frame,
    load_validation_cases,
    locate_validation_template,
    pick_row,
    select_test_cases,
    validation_cases_to_rows,
)
from scripts.helpers.notebook_bootstrap import bootstrap_demo_notebook
from scripts.helpers.notebook_display import dataframe_preview, demo_error, markdown_summary, test_case_preview
from scripts.helpers.result_writer import (
    create_run_artifact_paths,
    write_json,
    write_markdown,
    write_queries,
    write_query_results,
    write_rows_to_csv,
)

context = bootstrap_demo_notebook(REPO_ROOT)
display(Markdown(markdown_summary('Notebook Context', {
    'Repo Root': context.repo_root,
    'Auth Mode': context.notebook_context.settings.auth_mode,
    'Config Environment': context.notebook_context.settings.environment_name,
})))


### Notebook Context

- **Repo Root**: C:\Point\2026\Speaker\PBIPxCopilot\powerbi_demo_PBIPxGHCopilot
- **Auth Mode**: service_principal
- **Config Environment**: dev

## 1. Configure the Demo

Edit the variables in the next cell if you want to steer the demo toward a known workspace, dataset, or subset of validation cases. If you leave them blank, the notebook falls back to the first available match.


In [2]:
AUTH_MODE = None
USE_DEVICE_CODE = True

WORKSPACE_NAME_CONTAINS = ''
WORKSPACE_INDEX = 0
DATASET_NAME_CONTAINS = ''
DATASET_INDEX = 0

TEMPLATE_OVERRIDE = None
STATUS_FILTER = 'draft'
REVIEW_STATUS_FILTER = ''
PRIORITY_FILTER = 'high'
MEASURE_NAME_FILTER = ''
SCENARIO_TYPE_FILTER = ''
SELECTED_TEST_IDS = []
CASE_LIMIT = 5

RESULTS_BASE_DIR = REPO_ROOT / 'test-results'


## 2. Discover Workspaces and Datasets

This section authenticates with Power BI, lists visible workspaces, and picks a dataset for the rest of the notebook. The chosen workspace and dataset are displayed explicitly so the audience can follow the context.


In [3]:
client = context.create_client(auth_mode=AUTH_MODE, use_device_code=USE_DEVICE_CODE)

workspace_frame = get_workspaces_frame(client)
display(Markdown(markdown_summary('Workspace Discovery', {
    'Workspace Count': len(workspace_frame),
    'Selection Hint': WORKSPACE_NAME_CONTAINS or '<first row>',
})))
display(dataframe_preview(workspace_frame, limit=10))

selected_workspace = pick_row(workspace_frame, preferred_name=WORKSPACE_NAME_CONTAINS, index=WORKSPACE_INDEX)
if selected_workspace is None:
    raise RuntimeError('No workspaces were returned for the current identity.')

workspace_id = selected_workspace['id']
dataset_frame = get_datasets_frame(client, workspace_id)
display(Markdown(markdown_summary('Selected Workspace', {
    'Workspace Name': selected_workspace.get('name', ''),
    'Workspace Id': workspace_id,
    'Dataset Count': len(dataset_frame),
})))
display(dataframe_preview(dataset_frame, limit=10))

selected_dataset = pick_row(dataset_frame, preferred_name=DATASET_NAME_CONTAINS, index=DATASET_INDEX)
if selected_dataset is None:
    raise RuntimeError('No datasets were returned for the selected workspace.')

dataset_id = selected_dataset['id']
dataset_name = selected_dataset.get('name', '')
reports_frame = get_reports_frame(client, workspace_id)

display(Markdown(markdown_summary('Selected Dataset', {
    'Dataset Name': dataset_name,
    'Dataset Id': dataset_id,
    'Configured By': selected_dataset.get('configuredBy', ''),
    'Reports In Workspace': len(reports_frame),
})))


2026-03-14 11:06:30,230 | INFO | src.auth.service_principal | Service principal token acquired successfully.


### Workspace Discovery

- **Workspace Count**: 1
- **Selection Hint**: <first row>

,id,name,type,isReadOnly,isOnDedicatedCapacity
0,3155f514-76c4-4eea-9684-4a5b240c88ee,Demo_Powerplatform_2026_Prod,Workspace,False,True


### Selected Workspace

- **Workspace Name**: Demo_Powerplatform_2026_Prod
- **Workspace Id**: 3155f514-76c4-4eea-9684-4a5b240c88ee
- **Dataset Count**: 1

,id,name,configuredBy,isRefreshable,targetStorageMode
0,76fbbb23-6d54-4c81-9bc4-6b7b8faed659,demo_dataset,Charnrit@bcscdev.onmicrosoft.com,True,Abf


### Selected Dataset

- **Dataset Name**: demo_dataset
- **Dataset Id**: 76fbbb23-6d54-4c81-9bc4-6b7b8faed659
- **Configured By**: Charnrit@bcscdev.onmicrosoft.com
- **Reports In Workspace**: 1

## 3. Load the Measure Validation Template

The notebook first looks for a dataset-specific template. If that is missing, it falls back to the shared template or generated candidate file. The fallback is shown clearly so the audience can see whether the notebook is working from approved content or inferred draft content.


In [4]:
template_selection = locate_validation_template(context.repo_root, dataset_name)
if TEMPLATE_OVERRIDE:
    template_selection = type(template_selection)(Path(TEMPLATE_OVERRIDE), 'manual_override', 'Using an explicit template override path.')

validation_frame = load_validation_cases(template_selection.path, dataset_name)
filtered_cases = filter_validation_cases(
    validation_frame,
    status=STATUS_FILTER,
    review_status=REVIEW_STATUS_FILTER,
    priority=PRIORITY_FILTER,
    measure_name=MEASURE_NAME_FILTER,
    scenario_type=SCENARIO_TYPE_FILTER,
)
selected_cases = select_test_cases(filtered_cases, selected_test_ids=SELECTED_TEST_IDS, limit=CASE_LIMIT)

display(Markdown(markdown_summary('Template Selection', {
    'Template Path': template_selection.path or '<missing>',
    'Template Source': template_selection.source,
    'Note': template_selection.note,
    'Available Cases': len(validation_frame),
    'Filtered Cases': len(filtered_cases),
    'Selected Cases': len(selected_cases),
})))

if validation_frame.empty:
    display(Markdown('**No validation cases were available for the selected dataset.**'))
else:
    display(test_case_preview(filtered_cases))


### Template Selection

- **Template Path**: C:\Point\2026\Speaker\PBIPxCopilot\powerbi_demo_PBIPxGHCopilot\tests\measure-validation\templates\measure_validation_template.csv
- **Template Source**: template
- **Note**: Loaded 24 row(s) for dataset 'demo_dataset'.
- **Available Cases**: 24
- **Filtered Cases**: 24
- **Selected Cases**: 5

,test_id,status,review_status,priority,table_name,measure_name,scenario_type,report_name,page_name,visual_name
0,mv_70ee614a35,draft,inferred,high,Fact Sales,Average Order Value,divide_by_zero,demo_dataset,Executive Overview,f26d7a00b6c48e56e7ec
1,mv_28072a574a,draft,inferred,high,Fact Sales,Average Order Value,grand_total_behavior,demo_dataset,Executive Overview,f26d7a00b6c48e56e7ec
2,mv_3cb6832517,draft,inferred,high,Fact Sales,Average Order Value,happy_path,demo_dataset,Executive Overview,f26d7a00b6c48e56e7ec
3,mv_edd3350c89,draft,inferred,high,Fact Sales,Average Order Value,regression,demo_dataset,Executive Overview,f26d7a00b6c48e56e7ec
4,mv_5837eb1bfc,draft,inferred,high,Fact Sales,Gross Margin,grand_total_behavior,demo_dataset,Executive Overview,2797f9375247dd4787c3
5,mv_631ee7b8aa,draft,inferred,high,Fact Sales,Gross Margin,happy_path,demo_dataset,Executive Overview,2797f9375247dd4787c3
6,mv_6dd7577dde,draft,inferred,high,Fact Sales,Gross Margin,negative_values,demo_dataset,Executive Overview,2797f9375247dd4787c3
7,mv_17a2a4fdfb,draft,inferred,high,Fact Sales,Gross Margin,regression,demo_dataset,Executive Overview,2797f9375247dd4787c3
8,mv_1c6277a41e,draft,inferred,high,Fact Sales,Gross Margin %,divide_by_zero,,,
9,mv_d1d8d5559c,draft,inferred,high,Fact Sales,Gross Margin %,format_consistency,,,


## 4. Execute Selected Test Cases

Each test case is executed individually. The notebook keeps the DAX query visible, shows a small result preview, and records a lightweight `pass` / `needs_review` / `error` outcome without pretending that machine inference replaces human review.


In [5]:
execution_results = []

for test_case in validation_cases_to_rows(selected_cases):
    result = execute_validation_case(client, workspace_id, dataset_id, test_case)
    execution_results.append(result)

    display(Markdown(f"### {result['test_id']} :: {result['measure_name']}"))

    scenario_lines = "\n".join([
        f"- Scenario: `{result['scenario_type']}`",
        f"- Outcome: `{result['outcome']}`",
        f"- Expected Behavior: {result['expected_behavior']}",
    ])
    display(Markdown(scenario_lines))

    dax_block = "```sql\n" + result['dax_query'] + "\n```"
    display(Markdown(dax_block))

    if result['error']:
        display(Markdown(f"**Query Error**: `{result['error']}`"))
    elif result['rows']:
        display(pd.DataFrame(result['rows']).head(10))
    else:
        display(Markdown('_Query returned no rows._'))

execution_summary = build_execution_summary_frame(execution_results)
display(Markdown(markdown_summary('Execution Summary', {
    'Executed Cases': len(execution_results),
    'Pass Count': int((execution_summary['outcome'] == 'pass').sum()) if not execution_summary.empty else 0,
    'Needs Review Count': int((execution_summary['outcome'] == 'needs_review').sum()) if not execution_summary.empty else 0,
    'Error Count': int((execution_summary['outcome'] == 'error').sum()) if not execution_summary.empty else 0,
})))
display(execution_summary)


### mv_70ee614a35 :: Average Order Value

- Scenario: `divide_by_zero`
- Outcome: `needs_review`
- Expected Behavior: The measure returns BLANK or the defined safe fallback instead of an error.

```sql
-- test_id: mv_70ee614a35
-- measure: Fact Sales[Average Order Value]
-- intended_filter_context: Apply a slice where the denominator measure resolves to zero or blank.
EVALUATE
ROW("Measure", "Average Order Value", "Value", [Average Order Value])
```

,[Measure],[Value]
0,Average Order Value,9000.0


### mv_28072a574a :: Average Order Value

...

- Scenario: `grand_total_behavior`
- Outcome: `needs_review`
- Expected Behavior: Totals align with the intended business rule rather than accidental aggregation.

```sql
-- test_id: mv_5837eb1bfc
-- measure: Fact Sales[Gross Margin]
-- intended_filter_context: Compare row-level or point-level values to the aggregate total on Executive Overview.
EVALUATE
ROW("Measure", "Gross Margin", "Value", [Gross Margin])
```

,[Measure],[Value]
0,Gross Margin,80600


### Execution Summary

- **Executed Cases**: 5
- **Pass Count**: 0
- **Needs Review Count**: 5
- **Error Count**: 0

,test_id,measure_name,scenario_type,row_count,outcome,error
0,mv_70ee614a35,Average Order Value,divide_by_zero,1,needs_review,
1,mv_28072a574a,Average Order Value,grand_total_behavior,1,needs_review,
2,mv_3cb6832517,Average Order Value,happy_path,1,needs_review,
3,mv_edd3350c89,Average Order Value,regression,1,needs_review,
4,mv_5837eb1bfc,Gross Margin,grand_total_behavior,1,needs_review,


## 5. Review Report Context

This section shows workspace report metadata and, when the local PBIP demo report is available, measure-to-page-to-visual mappings for the selected measures. If report rendering is not practical, this metadata summary still gives clear report relevance.


In [6]:
selected_measure_names = selected_cases['measure_name'].tolist() if not selected_cases.empty else []
report_context = build_report_context(context.repo_root, reports_frame, dataset_id, dataset_name, selected_measure_names)
report_summary_markdown = build_report_summary_markdown(report_context)

display(Markdown(report_summary_markdown))
if report_context['api_reports']:
    display(pd.DataFrame(report_context['api_reports']))
if report_context['local_measure_usage']:
    display(pd.DataFrame(report_context['local_measure_usage']))


# Report Summary

## Workspace Reports
- demo_dataset (0e0a977e-7733-4fb3-9928-f01ebdec4648)

## Local PBIP Measure Usage
- Executive Overview / 2797f9375247dd4787c3 -> Fact Sales[Gross Margin]
- Executive Overview / f26d7a00b6c48e56e7ec -> Fact Sales[Average Order Value]

,id,name,datasetId,webUrl,embedUrl
0,0e0a977e-7733-4fb3-9928-f01ebdec4648,demo_dataset,76fbbb23-6d54-4c81-9bc4-6b7b8faed659,https://app.powerbi.com/groups/3155f514-76c4-4...,https://app.powerbi.com/reportEmbed?reportId=0...


,dataset_name,report_name,page_name,visual_name,visual_type,table_name,measure_name,usage_role,page_id,visual_id,query_ref,is_high_visibility
0,demo_dataset,demo_dataset,Executive Overview,2797f9375247dd4787c3,lineChart,Fact Sales,Gross Margin,Y,f1be4a98bb76609d1c08,2797f9375247dd4787c3,Fact Sales.Gross Margin,True
1,demo_dataset,demo_dataset,Executive Overview,f26d7a00b6c48e56e7ec,lineChart,Fact Sales,Average Order Value,Y,f1be4a98bb76609d1c08,f26d7a00b6c48e56e7ec,Fact Sales.Average Order Value,True


## 6. Save a Demo-Friendly Result Package

The final section writes a clean timestamped run folder under `test-results/`. This keeps the notebook useful for demos and follow-up review without turning it into a larger application.


In [7]:
artifact_paths = create_run_artifact_paths(RESULTS_BASE_DIR, dataset_name)
selected_case_rows = validation_cases_to_rows(selected_cases)
query_rows = [
    {
        'test_id': result['test_id'],
        'measure_name': result['measure_name'],
        'dax_query': result['dax_query'],
    }
    for result in execution_results
]
run_summary = {
    'workspace': selected_workspace,
    'dataset': selected_dataset,
    'template_path': str(template_selection.path) if template_selection.path else '',
    'template_source': template_selection.source,
    'selected_case_count': len(selected_case_rows),
    'executed_case_count': len(execution_results),
    'execution_outcomes': execution_summary.to_dict(orient='records') if not execution_summary.empty else [],
}

write_json(artifact_paths.run_summary_path, run_summary)
write_rows_to_csv(artifact_paths.selected_test_cases_path, selected_case_rows)
write_queries(artifact_paths.executed_queries_path, query_rows)
write_query_results(artifact_paths.query_results_dir, execution_results)
write_json(artifact_paths.report_metadata_path, report_context)
write_markdown(artifact_paths.report_summary_path, report_summary_markdown)

display(Markdown(markdown_summary('Artifacts Saved', {
    'Run Folder': artifact_paths.run_root,
    'Run Summary': artifact_paths.run_summary_path.name,
    'Selected Test Cases': artifact_paths.selected_test_cases_path.name,
    'Executed Queries': artifact_paths.executed_queries_path.name,
    'Query Result Files': len(execution_results),
    'Report Summary': artifact_paths.report_summary_path.name,
})))


### Artifacts Saved

- **Run Folder**: C:\Point\2026\Speaker\PBIPxCopilot\powerbi_demo_PBIPxGHCopilot\test-results\demo-dataset\2026-03-14_110631
- **Run Summary**: run_summary.json
- **Selected Test Cases**: selected_test_cases.csv
- **Executed Queries**: executed_queries.sql.txt
- **Query Result Files**: 5
- **Report Summary**: report_summary.md

## Next Step

If you want a more curated presentation, set `SELECTED_TEST_IDS` to a small list of reviewed test cases and rerun sections 3 through 6. That produces a cleaner narrative while preserving the same saved artifact structure.
